# 01 · Data Ingestion from AGSI+

Fetch EU gas storage data. Run cells **in order**.

---

In [ ]:
import sys, os, warnings, pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"✅ Project root: {_c}")
        break

In [ ]:
from src.agsi_client import AGSIClient

# ── Paste your AGSI API key here ──────────────────────────────────────────────
# Get free key at: https://agsi.gie.eu → Register → My Account → API Key
API_KEY = "paste_your_agsi_key_here"   # ← replace this

client = AGSIClient(api_key=API_KEY, ssl_verify=False)
print("✅ AGSIClient ready")

## 1.1 · Test — Single Country (DE)

In [ ]:
import pandas as pd

df_de = client.get_country("DE", start="2024-01-01")
print(f"✅ Germany: {len(df_de)} rows")
print(f"   Latest fill: {df_de['full'].dropna().iloc[-1]:.1f}%")
df_de.tail(3)

## 1.2 · EU Aggregate

In [ ]:
EU_COUNTRIES = ["DE","FR","IT","NL","AT","BE","ES","PL","CZ","HU"]
frames = []
for c in EU_COUNTRIES:
    df_c = client.get_country(c, start="2020-01-01")
    if not df_c.empty:
        frames.append(df_c)
        print(f"✅ {c}: {len(df_c)} rows, fill: {df_c['full'].dropna().iloc[-1]:.1f}%")

num_cols = [c for c in ["gasInStorage","injection","withdrawal","workingGasVolume",
                         "injectionCapacity","withdrawalCapacity"] if c in frames[0].columns]
df_eu = pd.concat(frames).groupby(level=0)[num_cols].sum()
df_eu["full"] = (df_eu["gasInStorage"]/df_eu["workingGasVolume"]*100).round(2)
df_eu["country"] = "EU"
df_eu.index = pd.to_datetime(df_eu.index)
df_eu = df_eu.sort_index()

fill_pct    = float(df_eu["gasInStorage"].dropna().iloc[-1]/df_eu["workingGasVolume"].dropna().iloc[-1]*100)
current_gwh = float(df_eu["gasInStorage"].dropna().iloc[-1])*1000
capacity    = float(df_eu["workingGasVolume"].dropna().iloc[-1])*1000

print(f"\n✅ EU: {len(df_eu)} rows | fill: {fill_pct:.1f}%")
df_eu.tail(3)

## 1.3 · Top-5 Countries

In [ ]:
df_countries = client.get_multiple_countries(["DE","FR","IT","NL","AT"], start="2020-01-01")
print(f"✅ {len(df_countries)} rows across {df_countries['country'].nunique()} countries")
df_countries.groupby("country")[["full"]].last().sort_values("full", ascending=False)

## 1.4 · Full History (2011–present)

In [ ]:
EU_COUNTRIES_FULL = ["DE","FR","IT","NL","AT","BE","ES","PL","CZ","HU"]
frames_full = []
for c in EU_COUNTRIES_FULL:
    df_c = client.get_country(c, start="2011-01-01")
    if not df_c.empty: frames_full.append(df_c)
    print(f"✅ {c}: {len(df_c)} rows")

df_eu_full = pd.concat(frames_full).groupby(level=0)[num_cols].sum()
df_eu_full["full"] = (df_eu_full["gasInStorage"]/df_eu_full["workingGasVolume"]*100).round(2)
df_eu_full.index = pd.to_datetime(df_eu_full.index)
df_eu_full = df_eu_full.sort_index()
print(f"\n✅ Full history: {len(df_eu_full)} rows | {df_eu_full.index.min().date()} → {df_eu_full.index.max().date()}")

## 1.5 · Save to Parquet

In [ ]:
Path("data/processed").mkdir(parents=True, exist_ok=True)
df_eu_full.to_parquet("data/processed/eu_aggregate_full.parquet")
df_countries.to_parquet("data/processed/top5_countries.parquet")
print("✅ Saved:")
print(f"   eu_aggregate_full.parquet : {df_eu_full.shape}")
print(f"   top5_countries.parquet    : {df_countries.shape}")

## 1.6 · Sanity Check

In [ ]:
import plotly.graph_objects as go
latest = df_eu_full["full"].dropna().iloc[-1]
latest_date = df_eu_full["full"].dropna().index[-1]
print(f"Latest EU fill: {latest:.1f}% as of {latest_date.date()}")
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_eu_full.index, y=df_eu_full["full"], name="EU Fill %",
                          line=dict(color="#2E75B6", width=1.5)))
fig.add_hline(y=90, line_dash="dot", line_color="purple", annotation_text="90% Target")
fig.update_layout(title="EU Gas Storage Fill Rate", yaxis=dict(ticksuffix="%", range=[0,105]),
                  xaxis_title="Date", template="plotly_white", hovermode="x unified")
fig.show()

## 2.0 · Daily Incremental Update

Run this section daily to fetch only new data — much faster than re-fetching everything.

In [ ]:
def update_country_parquet(client, country, parquet_path):
    import pandas as pd
    from pathlib import Path
    path = Path(parquet_path)
    if path.exists():
        existing = pd.read_parquet(path)
        existing.index = pd.to_datetime(existing.index)
        last_date = existing.index.max()
        start = (last_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        print(f"  {country}: last={last_date.date()} | fetching from {start}")
    else:
        existing = pd.DataFrame()
        start = "2011-01-01"
        print(f"  {country}: full fetch from {start}")
    new_data = client.get_country(country, start=start, use_cache=False)
    if new_data.empty:
        print(f"  {country}: no new data")
        return existing
    print(f"  {country}: +{len(new_data)} new rows")
    combined = pd.concat([existing, new_data]) if not existing.empty else new_data
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()
    combined.to_parquet(path)
    return combined

# Run daily update
EU_COUNTRIES = ["DE","FR","IT","NL","AT","BE","ES","PL","CZ","HU"]
Path("data/processed/countries").mkdir(parents=True, exist_ok=True)
print("Updating country parquets...")
for country in EU_COUNTRIES:
    update_country_parquet(client, country, f"data/processed/countries/{country.lower()}.parquet")

# Rebuild EU aggregate
print("\nRebuilding EU aggregate...")
frames_upd = []
for country in EU_COUNTRIES:
    p = Path(f"data/processed/countries/{country.lower()}.parquet")
    if p.exists():
        df_c = pd.read_parquet(p)
        df_c["country"] = country
        frames_upd.append(df_c)

num_cols = [c for c in ["gasInStorage","injection","withdrawal","workingGasVolume",
                         "injectionCapacity","withdrawalCapacity"] if c in frames_upd[0].columns]
df_eu_updated = pd.concat(frames_upd).groupby(level=0)[num_cols].sum()
df_eu_updated["full"] = (df_eu_updated["gasInStorage"]/df_eu_updated["workingGasVolume"]*100).round(2)
df_eu_updated.index = pd.to_datetime(df_eu_updated.index)
df_eu_updated = df_eu_updated.sort_index()
df_eu_updated.to_parquet("data/processed/eu_aggregate_full.parquet")
print(f"\n✅ Updated: {len(df_eu_updated)} rows | latest fill: {df_eu_updated['full'].dropna().iloc[-1]:.1f}%")